# Jump-Diffusion Research Notebook

This notebook explores a possible Phase 2 extension of the Probabilistic Price Path Engine.

The current dashboard supports:

- GBM Monte Carlo simulation
- historical bootstrap simulation
- analytical GBM terminal benchmarking
- pathwise TP/SL probability estimation

This notebook investigates whether a jump-diffusion model can better capture occasional large intraday moves.

The goal is research only. The model should be tested here before being added to the Streamlit dashboard.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from data_loader import load_market_data
from simulator import estimate_mu_sigma, compute_log_returns, simulate_gbm_paths, simulate_bootstrap_paths
from probability_engine import pathwise_tp_sl_metrics
from charts import make_price_path_figure

In [ ]:
DATA_SOURCE = "synthetic"   # "synthetic", "mt5", or "csv"

SYMBOL = "US100.cash"
TIMEFRAME = "M1"
BARS = 1000

CSV_PATH = PROJECT_ROOT / "data" / "historical" / "sample.csv"

TP_POINTS = 29.0
SL_POINTS = 29.0

HORIZON = 20
N_PATHS = 10_000
VOL_WINDOW = 90

DRIFT_MODE = "zero"
DIRECTION = "long"

SEED = 42

In [ ]:
if DATA_SOURCE == "csv":
    df = load_market_data(
        source="csv",
        csv_path=CSV_PATH,
    )
else:
    df = load_market_data(
        source=DATA_SOURCE,
        symbol=SYMBOL,
        timeframe=TIMEFRAME,
        bars=BARS,
        synthetic_start_price=21500.0,
        synthetic_seed=SEED,
        synthetic_freq="1min",
    )

# Use closed-candle logic for MT5 research.
if DATA_SOURCE == "mt5" and len(df) > VOL_WINDOW + 21:
    model_df = df.iloc[:-1].copy()
else:
    model_df = df.copy()

model_df.tail()

## Log Return Analysis

Jump-diffusion modelling starts by separating normal candle-to-candle noise from unusually large returns.

First, calculate log returns:

$$
r_t = \log\left(\frac{S_t}{S_{t-1}}\right)
$$

Then estimate recent drift and volatility using the same engine functions as the dashboard.

In [ ]:
current_price = float(model_df["Close"].iloc[-1])

mu, sigma, log_returns = estimate_mu_sigma(
    model_df["Close"],
    window=VOL_WINDOW,
    drift_mode=DRIFT_MODE,
)

recent_returns = log_returns.dropna().tail(VOL_WINDOW)

return_summary = {
    "current_price": current_price,
    "mu_per_candle": mu,
    "sigma_per_candle": sigma,
    "vol_window": VOL_WINDOW,
    "mean_recent_return": float(recent_returns.mean()),
    "std_recent_return": float(recent_returns.std(ddof=1)),
    "min_recent_return": float(recent_returns.min()),
    "max_recent_return": float(recent_returns.max()),
}

return_summary

## Jump Detection Rule

For this first research version, define a jump as a return whose absolute size is larger than a multiple of recent volatility:

$$
|r_t - \bar{r}| > k\sigma
$$

where:

- $r_t$ is the candle log return
- $\bar{r}$ is the recent mean return
- $\sigma$ is recent return volatility
- $k$ is the jump threshold multiplier

This is a simple threshold-based jump detector. It is not final model logic yet.

In [ ]:
JUMP_THRESHOLD = 2.5

recent_mean = float(recent_returns.mean())
recent_std = float(recent_returns.std(ddof=1))

jump_mask = (recent_returns - recent_mean).abs() > JUMP_THRESHOLD * recent_std
jump_returns = recent_returns[jump_mask]
normal_returns = recent_returns[~jump_mask]

jump_stats = {
    "jump_threshold_sigma": JUMP_THRESHOLD,
    "n_recent_returns": len(recent_returns),
    "n_jumps": int(jump_mask.sum()),
    "jump_intensity_per_candle": float(jump_mask.mean()),
    "normal_return_mean": float(normal_returns.mean()) if len(normal_returns) else np.nan,
    "normal_return_std": float(normal_returns.std(ddof=1)) if len(normal_returns) > 1 else np.nan,
    "jump_return_mean": float(jump_returns.mean()) if len(jump_returns) else 0.0,
    "jump_return_std": float(jump_returns.std(ddof=1)) if len(jump_returns) > 1 else 0.0,
}

jump_stats

In [ ]:
jump_table = (
    pd.DataFrame({
        "time": model_df["time"].iloc[-len(recent_returns):].values,
        "log_return": recent_returns.values,
        "is_jump": jump_mask.values,
    })
    .query("is_jump == True")
    .reset_index(drop=True)
)

jump_table

## Jump-Diffusion Simulation

A simple jump-diffusion return step can be written as:

$$
r_t = \left(\mu - \frac{1}{2}\sigma^2\right) + \sigma Z_t + J_t N_t
$$

where:

- $Z_t \sim \mathcal{N}(0,1)$ is the normal GBM shock
- $N_t \sim \text{Bernoulli}(\lambda)$ decides whether a jump occurs in that candle
- $J_t$ is the jump size sampled from detected historical jump returns
- $\lambda$ is the estimated jump intensity per candle

This is a research approximation of a Merton-style jump-diffusion process.

In [ ]:
def simulate_jump_diffusion_paths(
    current_price: float,
    mu: float,
    sigma: float,
    jump_intensity: float,
    jump_mean: float,
    jump_std: float,
    horizon: int,
    n_paths: int,
    seed: int | None = None,
) -> np.ndarray:
    rng = np.random.default_rng(seed)

    normal_shocks = rng.normal(0, 1, size=(n_paths, horizon))
    gbm_steps = (mu - 0.5 * sigma**2) + sigma * normal_shocks

    jump_occurs = rng.binomial(
        n=1,
        p=jump_intensity,
        size=(n_paths, horizon),
    )

    if jump_std > 0:
        jump_sizes = rng.normal(
            loc=jump_mean,
            scale=jump_std,
            size=(n_paths, horizon),
        )
    else:
        jump_sizes = np.full((n_paths, horizon), jump_mean)

    log_steps = gbm_steps + jump_occurs * jump_sizes
    log_paths = np.cumsum(log_steps, axis=1)

    prices = current_price * np.exp(log_paths)

    return np.column_stack([np.full(n_paths, current_price), prices])

In [ ]:
jump_paths = simulate_jump_diffusion_paths(
    current_price=current_price,
    mu=mu,
    sigma=sigma,
    jump_intensity=jump_stats["jump_intensity_per_candle"],
    jump_mean=jump_stats["jump_return_mean"],
    jump_std=jump_stats["jump_return_std"],
    horizon=HORIZON,
    n_paths=N_PATHS,
    seed=SEED,
)

jump_metrics = pathwise_tp_sl_metrics(
    paths=jump_paths,
    entry_price=current_price,
    direction=DIRECTION,
    tp_points=TP_POINTS,
    sl_points=SL_POINTS,
)

jump_metrics

In [ ]:
gbm_paths = simulate_gbm_paths(
    current_price=current_price,
    mu=mu,
    sigma=sigma,
    horizon=HORIZON,
    n_paths=N_PATHS,
    seed=SEED,
)

gbm_metrics = pathwise_tp_sl_metrics(
    paths=gbm_paths,
    entry_price=current_price,
    direction=DIRECTION,
    tp_points=TP_POINTS,
    sl_points=SL_POINTS,
)

bootstrap_paths = simulate_bootstrap_paths(
    current_price=current_price,
    historical_returns=log_returns,
    horizon=HORIZON,
    n_paths=N_PATHS,
    sample_window=VOL_WINDOW,
    seed=SEED,
)

bootstrap_metrics = pathwise_tp_sl_metrics(
    paths=bootstrap_paths,
    entry_price=current_price,
    direction=DIRECTION,
    tp_points=TP_POINTS,
    sl_points=SL_POINTS,
)

model_comparison = pd.DataFrame([
    {
        "Model": "GBM",
        "P(TP before SL)": gbm_metrics["p_tp_first"],
        "P(SL before TP)": gbm_metrics["p_sl_first"],
        "P(neither)": gbm_metrics["p_neither"],
        "Expected Move": gbm_metrics["expected_move"],
        "5%": gbm_metrics["p5"],
        "95%": gbm_metrics["p95"],
    },
    {
        "Model": "Bootstrap",
        "P(TP before SL)": bootstrap_metrics["p_tp_first"],
        "P(SL before TP)": bootstrap_metrics["p_sl_first"],
        "P(neither)": bootstrap_metrics["p_neither"],
        "Expected Move": bootstrap_metrics["expected_move"],
        "5%": bootstrap_metrics["p5"],
        "95%": bootstrap_metrics["p95"],
    },
    {
        "Model": "Jump-Diffusion",
        "P(TP before SL)": jump_metrics["p_tp_first"],
        "P(SL before TP)": jump_metrics["p_sl_first"],
        "P(neither)": jump_metrics["p_neither"],
        "Expected Move": jump_metrics["expected_move"],
        "5%": jump_metrics["p5"],
        "95%": jump_metrics["p95"],
    },
])

model_comparison

In [ ]:
fig = make_price_path_figure(
    df=model_df,
    paths=jump_paths,
    metrics=jump_metrics,
    max_paths=120,
)

fig.show()

## Interpretation

The jump-diffusion model adds a jump component to the usual GBM return process.

This means the model can generate paths where price moves normally most of the time, but occasionally experiences a larger jump-like move.

This is useful for intraday index data because sharp candles may occur around:

- market open
- economic news
- liquidity shocks
- fast momentum bursts
- stop runs or volatility expansions

However, this first version is only a research approximation. The jump threshold, jump intensity, and jump-size distribution need further validation before being used in the live dashboard.

In [ ]:
diagnostics = pd.DataFrame([
    {
        "Metric": "Jump threshold sigma",
        "Value": jump_stats["jump_threshold_sigma"],
    },
    {
        "Metric": "Recent returns analysed",
        "Value": jump_stats["n_recent_returns"],
    },
    {
        "Metric": "Detected jumps",
        "Value": jump_stats["n_jumps"],
    },
    {
        "Metric": "Jump intensity per candle",
        "Value": jump_stats["jump_intensity_per_candle"],
    },
    {
        "Metric": "Normal return mean",
        "Value": jump_stats["normal_return_mean"],
    },
    {
        "Metric": "Normal return std",
        "Value": jump_stats["normal_return_std"],
    },
    {
        "Metric": "Jump return mean",
        "Value": jump_stats["jump_return_mean"],
    },
    {
        "Metric": "Jump return std",
        "Value": jump_stats["jump_return_std"],
    },
])

diagnostics

## Research Notes

Key questions before adding this to the dashboard:

1. Does jump-diffusion produce materially different TP/SL probabilities from GBM and bootstrap?
2. Does it improve realism, or does it over-amplify extreme moves?
3. Is the jump threshold stable across different sessions and timeframes?
4. Should jumps be signed, absolute, or session-conditioned?
5. Should jump intensity be estimated from a rolling window or a longer historical sample?

Possible future improvement:

- classify market regimes before estimating jump intensity
- estimate separate positive and negative jump distributions
- condition jumps on session/time-of-day
- compare jump-diffusion outputs against realised TP/SL outcomes

In [ ]:
output_dir = PROJECT_ROOT / "reports" / "logs"
output_dir.mkdir(parents=True, exist_ok=True)

comparison_path = output_dir / "jump_diffusion_model_comparison.csv"
diagnostics_path = output_dir / "jump_diffusion_diagnostics.csv"

model_comparison.to_csv(comparison_path, index=False)
diagnostics.to_csv(diagnostics_path, index=False)

comparison_path, diagnostics_path

## Next Steps

This notebook should remain a research sandbox until the jump-diffusion model is validated.

If the model is useful, the next implementation step would be:

1. move `simulate_jump_diffusion_paths()` into `src/simulator.py`
2. add jump parameter estimation functions
3. add `Jump-Diffusion` to the Streamlit model dropdown
4. compare live dashboard outputs against GBM and bootstrap
5. document the methodology in the README